In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer



In [29]:
# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")



All libraries imported successfully!


In [ ]:

# ============================================================
# STEP 2: Load Dataset
# ============================================================
df = pd.read_csv("housing.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())


# ============================================================
# STEP 3: Check Missing Values
# ============================================================
print("\nMissing Values in each column:")
print(df.isnull().sum())


# ============================================================
# STEP 4: Handle Missing Values
# ============================================================
# Fill missing values in 'total_bedrooms' with median
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

print("\nMissing values after handling:", df.isnull().sum().sum())


# ============================================================
# STEP 5: Separate Features and Target
# ============================================================
y = df['median_house_value']
X = df.drop('median_house_value', axis=1)

print("\nFeatures:", X.columns.tolist())
print("Target: median_house_value")


# ============================================================
# STEP 6: Train Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# ============================================================
# STEP 7: Preprocessing (Scaling + Encoding)
# ============================================================
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

print("\nPreprocessing done!")


# ============================================================
# STEP 8: Train and Compare Multiple Models
# ============================================================
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

print("\n" + "=" * 60)
print(f"{'Model':<25} | {'MAE':<10} | {'RMSE':<10} | {'R2':<10}")
print("=" * 60)

for name, model in models.items():

    # Create pipeline
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train the model
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

    print(f"{name:<25} | {mae:<10.2f} | {rmse:<10.2f} | {r2:<10.4f}")

print("=" * 60)


# ============================================================
# STEP 9: Compare Models in a Table
# ============================================================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("\nMODEL COMPARISON (Sorted by R2):")
print(results_df)


# ============================================================
# STEP 10: Select Best Model
# ============================================================
best_model_name = results_df.iloc[0]['Model']
best_r2 = results_df.iloc[0]['R2']

print("\nBest Model:", best_model_name)
print("Best R2 Score:", best_r2)


# ============================================================
# STEP 11: Hyperparameter Tuning (Random Forest)
# ============================================================
print("\nPerforming Hyperparameter Tuning...")

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None]
}

grid = GridSearchCV(rf_pipe, params, cv=3, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best R2 Score:", grid.best_score_)


# ============================================================
# STEP 12: Final Evaluation
# ============================================================
y_pred_final = grid.predict(X_test)

print("\nFINAL MODEL PERFORMANCE:")
print("MAE  :", mean_absolute_error(y_test, y_pred_final))
print("MSE  :", mean_squared_error(y_test, y_pred_final))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_final)))
print("R2   :", r2_score(y_test, y_pred_final))


# ============================================================
# STEP 13: Plot Actual vs Predicted
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted")
plt.show()

Dataset Shape: (20640, 10)

First 5 Rows:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

Missing Values in each column:
l